# 🚀 Network Latency Predictor — Cloud Training (Google Colab)This notebook runs the **full training pipeline** on Colab's GPU/CPU resources.**Models trained:** ARIMA, XGBoost (×5 sub-models), LSTM (PyTorch + Attention), Queuing Theory  **Output:** Trained model artifacts saved to Google Drive or downloaded as ZIP> ⚡ **Runtime → Change runtime type → T4 GPU** for fastest LSTM training

In [ ]:
# @title 1️⃣ Install & Verify Dependencies# Colab already has numpy, pandas, scipy, scikit-learn, torch.# Only install the extras this project needs (NO version pins).!pip install -q statsmodels xgboost loguru pydantic-settings python-dotenv joblibimport numpy as np, pandas as pd, scipy, sklearn, torchprint(f"NumPy:        {np.__version__}")print(f"Pandas:       {pd.__version__}")print(f"SciPy:        {scipy.__version__}")print(f"Scikit-learn: {sklearn.__version__}")print(f"PyTorch:      {torch.__version__}")print(f"CUDA:         {torch.cuda.is_available()}")if torch.cuda.is_available():    print(f"GPU:          {torch.cuda.get_device_name(0)}")import statsmodels, xgboostprint(f"Statsmodels:  {statsmodels.__version__}")print(f"XGBoost:      {xgboost.__version__}")print("\n✅ All dependencies ready")

In [ ]:
# @title 2️⃣ Mount Google Drive (Optional — for persistent model storage)MOUNT_DRIVE = True  # @param {type:"boolean"}if MOUNT_DRIVE:    from google.colab import drive    drive.mount('/content/drive')    DRIVE_DIR = '/content/drive/MyDrive/netPredictor_models'    import os; os.makedirs(DRIVE_DIR, exist_ok=True)    print(f"✅ Drive mounted. Models will be saved to: {DRIVE_DIR}")else:    DRIVE_DIR = None    print("⏭️ Drive not mounted. Models will be downloaded as ZIP.")

In [ ]:
# @title 3️⃣ Create Project Structure on Colabimport osBASE = '/content/netPredictor'for d in ['config','collector','features','models','training','data','models/saved','scripts','logs']:    os.makedirs(f'{BASE}/{d}', exist_ok=True)    if d not in ['data','models/saved','logs']:        open(f'{BASE}/{d}/__init__.py','w').write('')os.chdir(BASE)import sys; sys.path.insert(0, BASE)print(f"✅ Project structure created at {BASE}")

In [ ]:
# @title 4️⃣ Write All Source Modulesimport osBASE = "/content/netPredictor"modules = {    "config/settings.py": '"""Application settings managed via pydantic-settings.\n\nAll configuration is loaded from environment variables or a `.env` file.\nNo value should ever be hard-coded in application code — reference\n``get_settings()`` instead.\n"""\n\nfrom __future__ import annotations\n\nfrom functools import lru_cache\nfrom typing import Dict, List\n\nfrom pydantic_settings import BaseSettings\n\n\nclass Settings(BaseSettings):\n    """Central configuration for every subsystem.\n\n    Attributes are grouped by subsystem and documented inline.\n    """\n\n    # ── Collector ───────────────────────────────────────────────\n    NETWORK_INTERFACE: str = "eth0"\n    PROBE_HOSTS: List[str] = ["8.8.8.8", "1.1.1.1", "208.67.222.222"]\n    PROBE_INTERVAL_SEC: float = 1.0\n    CAPTURE_FILTER: str = "icmp or tcp"\n\n    # ── Kafka ───────────────────────────────────────────────────\n    KAFKA_BOOTSTRAP_SERVERS: str = "localhost:9092"\n    KAFKA_TOPIC_RAW: str = "network.telemetry.raw"\n    KAFKA_TOPIC_FEATURES: str = "network.telemetry.features"\n    KAFKA_TOPIC_PREDICTIONS: str = "network.predictions"\n\n    # ── Feature windows (seconds) ──────────────────────────────\n    WINDOW_SHORT: int = 5\n    WINDOW_MEDIUM: int = 30\n    WINDOW_LONG: int = 300\n\n    # ── EWMA ────────────────────────────────────────────────────\n    EWMA_ALPHA: float = 0.3\n\n    # ── Model ───────────────────────────────────────────────────\n    MODEL_DIR: str = "./models/saved"\n    ENSEMBLE_WEIGHTS: Dict[str, float] = {\n        "arima": 0.2,\n        "xgboost": 0.4,\n        "lstm": 0.3,\n        "queuing": 0.1,\n    }\n\n    # ── API ─────────────────────────────────────────────────────\n    API_HOST: str = "0.0.0.0"\n    API_PORT: int = 8000\n\n    # ── MLflow ──────────────────────────────────────────────────\n    MLFLOW_TRACKING_URI: str = "http://localhost:5000"\n    MLFLOW_EXPERIMENT_NAME: str = "network_latency_prediction"\n\n    # ── Drift detection ────────────────────────────────────────\n    DRIFT_THRESHOLD_MAE: float = 5.0  # ms — retrain if MAE exceeds this\n    DRIFT_WINDOW_SIZE: int = 500  # samples to evaluate drift on\n\n    model_config = {"env_file": ".env", "env_file_encoding": "utf-8"}\n\n\n@lru_cache()\ndef get_settings() -> Settings:\n    """Return a cached singleton of application settings."""\n    return Settings()\n',    "config/logging_config.py": '"""Structured JSON logging configuration using loguru.\n\nCall ``setup_logging()`` once at process start-up to configure loguru\nwith structured JSON output, rotation, and appropriate log levels.\n"""\n\nfrom __future__ import annotations\n\nimport sys\n\nfrom loguru import logger\n\n\ndef setup_logging(level: str = "INFO", json_output: bool = True) -> None:\n    """Configure loguru for structured logging.\n\n    Args:\n        level: Minimum log level (DEBUG, INFO, WARNING, ERROR, CRITICAL).\n        json_output: If True, emit logs as JSON objects for machine parsing.\n    """\n    logger.remove()  # Remove default handler\n\n    if json_output:\n        logger.add(\n            sys.stderr,\n            level=level,\n            serialize=True,  # JSON output\n            backtrace=True,\n            diagnose=False,  # Avoid leaking local vars in production\n        )\n    else:\n        logger.add(\n            sys.stderr,\n            level=level,\n            format=(\n                "<green>{time:YYYY-MM-DD HH:mm:ss.SSS}</green> | "\n                "<level>{level: <8}</level> | "\n                "<cyan>{name}</cyan>:<cyan>{function}</cyan>:<cyan>{line}</cyan> — "\n                "<level>{message}</level>"\n            ),\n            backtrace=True,\n            diagnose=True,\n        )\n\n    # File sink with rotation\n    logger.add(\n        "logs/network_predictor_{time}.log",\n        level=level,\n        rotation="50 MB",\n        retention="7 days",\n        compression="gz",\n        serialize=json_output,\n    )\n\n    logger.info("Logging configured", level=level, json_output=json_output)\n',    "features/rolling_stats.py": '"""Rolling window statistics for time-series telemetry.\n\nComputes descriptive statistics over sliding windows using pandas\nrolling operations — no manual loops.\n"""\n\nfrom __future__ import annotations\n\nfrom typing import Any, Dict\n\nimport pandas as pd\ndef compute_rolling_stats(series: pd.Series, window: int) -> Dict[str, pd.Series]:\n    """Compute rolling descriptive statistics for a pandas Series.\n\n    All statistics are computed using ``pd.Series.rolling`` with\n    C-optimized aggregations — no Python-level ``apply`` loops.\n\n    Args:\n        series: Input time-series values (e.g. rtt_ms).\n        window: Rolling window size in number of samples.\n\n    Returns:\n        Dictionary mapping stat name to a pandas Series of the same\n        length as the input (leading NaNs for incomplete windows).\n        Keys: mean, std, min, max, p50, p95, p99, iqr, skewness, kurtosis.\n    """\n    roller = series.rolling(window=window, min_periods=1)\n\n    result: Dict[str, pd.Series] = {\n        "mean": roller.mean(),\n        "std": roller.std(),\n        "min": roller.min(),\n        "max": roller.max(),\n        "p50": roller.quantile(0.50),\n        "p95": roller.quantile(0.95),\n        "p99": roller.quantile(0.99),\n        "iqr": roller.quantile(0.75) - roller.quantile(0.25),\n        "skewness": roller.skew().fillna(0.0),\n        "kurtosis": roller.kurt().fillna(0.0),\n    }\n\n    return result\n',    "features/ewma.py": '"""Exponential Weighted Moving Average (EWMA) feature extractors.\n\nProvides EWMA smoothing, volatility (EWMA std), and gradient\n(rate-of-change of the smoothed signal) — the key congestion onset\nindicator.\n"""\n\nfrom __future__ import annotations\n\nimport pandas as pd\n\n\ndef compute_ewma(series: pd.Series, alpha: float) -> pd.Series:\n    """Compute the exponential weighted moving average.\n\n    Args:\n        series: Raw time-series values.\n        alpha: Smoothing factor in (0, 1). Higher = more responsive.\n\n    Returns:\n        EWMA-smoothed series.\n    """\n    return series.ewm(alpha=alpha, adjust=False).mean()\n\n\ndef compute_ewma_std(series: pd.Series, alpha: float) -> pd.Series:\n    """Compute the exponential weighted standard deviation (volatility).\n\n    Args:\n        series: Raw time-series values.\n        alpha: Smoothing factor in (0, 1).\n\n    Returns:\n        EWMA standard deviation series.\n    """\n    return series.ewm(alpha=alpha, adjust=False).std()\n\n\ndef compute_ewma_gradient(series: pd.Series, alpha: float) -> pd.Series:\n    """Compute the first derivative of the EWMA — rate of change.\n\n    This is the primary congestion onset signal: a positive gradient\n    with increasing magnitude suggests latency is rising.\n\n    Args:\n        series: Raw time-series values.\n        alpha: Smoothing factor for EWMA.\n\n    Returns:\n        First-order difference of the EWMA-smoothed series.\n    """\n    ewma = compute_ewma(series, alpha)\n    return ewma.diff()\n',    "features/fft_features.py": '"""FFT-based frequency-domain features for telemetry signals.\n\nExtracts dominant frequency, spectral entropy, and power-ratio\nfeatures that help detect rapid fluctuations indicative of congestion.\n"""\n\nfrom __future__ import annotations\n\nfrom typing import Any, Dict, List\n\nimport numpy as np\nimport pandas as pd\n\n\ndef _next_power_of_2(n: int) -> int:\n    """Return the next power of 2 >= n."""\n    return 1 << (n - 1).bit_length()\n\n\ndef compute_fft_features(\n    series: pd.Series, sample_rate: float = 1.0\n) -> Dict[str, Any]:\n    """Compute frequency-domain features from a time series via FFT.\n\n    The series is zero-padded to the next power of 2 for efficiency.\n\n    Args:\n        series: Time-series values (e.g., rtt_ms over a window).\n        sample_rate: Sampling rate in Hz (default 1.0 = 1 sample/sec).\n\n    Returns:\n        Dictionary of scalar features:\n        - dominant_frequency_hz: frequency with highest magnitude\n        - dominant_period_sec: 1 / dominant_frequency_hz\n        - spectral_entropy: entropy of the normalised power spectrum\n        - top3_frequencies: list of top-3 frequency components (Hz)\n        - spectral_power_ratio: ratio of power above 0.1 Hz to total\n    """\n    values = series.dropna().values.astype(np.float64)\n\n    if len(values) < 4:\n        return {\n            "dominant_frequency_hz": 0.0,\n            "dominant_period_sec": float("inf"),\n            "spectral_entropy": 0.0,\n            "top3_frequencies": [0.0, 0.0, 0.0],\n            "spectral_power_ratio": 0.0,\n        }\n\n    # Zero-mean the signal\n    values = values - np.mean(values)\n\n    # Zero-pad to next power of 2\n    padded_len = _next_power_of_2(len(values))\n    padded = np.zeros(padded_len)\n    padded[: len(values)] = values\n\n    # FFT\n    fft_vals = np.fft.rfft(padded)\n    magnitudes = np.abs(fft_vals)\n    freqs = np.fft.rfftfreq(padded_len, d=1.0 / sample_rate)\n\n    # Ignore DC component (index 0)\n    magnitudes_no_dc = magnitudes[1:]\n    freqs_no_dc = freqs[1:]\n\n    if len(magnitudes_no_dc) == 0 or magnitudes_no_dc.sum() == 0:\n        return {\n            "dominant_frequency_hz": 0.0,\n            "dominant_period_sec": float("inf"),\n            "spectral_entropy": 0.0,\n            "top3_frequencies": [0.0, 0.0, 0.0],\n            "spectral_power_ratio": 0.0,\n        }\n\n    # Dominant frequency\n    dominant_idx = np.argmax(magnitudes_no_dc)\n    dominant_freq = float(freqs_no_dc[dominant_idx])\n    dominant_period = 1.0 / dominant_freq if dominant_freq > 0 else float("inf")\n\n    # Top-3 frequencies\n    top3_indices = np.argsort(magnitudes_no_dc)[-3:][::-1]\n    top3_freqs: List[float] = [float(freqs_no_dc[i]) for i in top3_indices]\n    while len(top3_freqs) < 3:\n        top3_freqs.append(0.0)\n\n    # Spectral entropy\n    power = magnitudes_no_dc ** 2\n    total_power = power.sum()\n    if total_power > 0:\n        norm_power = power / total_power\n        # Avoid log(0)\n        norm_power = norm_power[norm_power > 0]\n        spectral_entropy = float(-np.sum(norm_power * np.log2(norm_power)))\n    else:\n        spectral_entropy = 0.0\n\n    # Spectral power ratio (power above 0.1 Hz / total)\n    high_freq_mask = freqs_no_dc > 0.1\n    high_freq_power = float(power[high_freq_mask].sum()) if high_freq_mask.any() else 0.0\n    spectral_power_ratio = high_freq_power / total_power if total_power > 0 else 0.0\n\n    return {\n        "dominant_frequency_hz": round(dominant_freq, 6),\n        "dominant_period_sec": round(dominant_period, 3),\n        "spectral_entropy": round(spectral_entropy, 4),\n        "top3_frequencies": [round(f, 6) for f in top3_freqs],\n        "spectral_power_ratio": round(float(spectral_power_ratio), 4),\n    }\n',    "features/gradient_estimator.py": '"""Finite-difference gradient estimator and congestion onset scorer.\n\nProvides numerical differentiation via ``numpy.gradient`` and a\ncomposite congestion-onset score combining RTT and utilization\ngradients.\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\n\ndef finite_difference_gradient(\n    series: pd.Series, order: int = 1\n) -> pd.Series:\n    """Compute the nth-order finite-difference gradient of a series.\n\n    Uses ``numpy.gradient`` which applies second-order central\n    differences in the interior and first/second-order forward/backward\n    differences at the boundaries.\n\n    Args:\n        series: Input time-series values.\n        order: Derivative order (1 = first derivative, 2 = second, …).\n\n    Returns:\n        Gradient series of the same length as input.\n    """\n    values = series.values.astype(np.float64)\n\n    for _ in range(order):\n        values = np.gradient(values)\n\n    return pd.Series(values, index=series.index, name=f"grad_order{order}")\n\n\ndef congestion_onset_score(\n    rtt_series: pd.Series,\n    utilization_series: pd.Series,\n) -> pd.Series:\n    """Compute a congestion-onset score combining RTT and utilization gradients.\n\n    The score is min-max normalised to [0.0, 1.0].  Values > 0.7\n    indicate congestion is likely developing.\n\n    Args:\n        rtt_series: RTT measurements (ms).\n        utilization_series: Network utilization proxy (e.g. bytes_per_sec).\n\n    Returns:\n        Series of scores in [0.0, 1.0].\n    """\n    rtt_grad = finite_difference_gradient(rtt_series, order=1)\n    util_grad = finite_difference_gradient(utilization_series, order=1)\n\n    # Combine: both positive gradients → congestion onset\n    combined = rtt_grad.clip(lower=0) + util_grad.clip(lower=0)\n\n    # Min-max normalisation\n    c_min = combined.min()\n    c_max = combined.max()\n\n    if c_max - c_min < 1e-12:\n        return pd.Series(\n            np.zeros(len(combined)), index=combined.index, name="congestion_onset_score"\n        )\n\n    normalised = (combined - c_min) / (c_max - c_min)\n    normalised = normalised.clip(0.0, 1.0)\n    normalised.name = "congestion_onset_score"\n    return normalised\n',    "features/pipeline.py": '"""Feature engineering pipeline — orchestrates all feature extractors.\n\nStateless and thread-safe: each ``transform()`` call creates no\nside-effects and can be called concurrently from multiple threads.\n"""\n\nfrom __future__ import annotations\n\nfrom typing import Dict, List\n\nimport numpy as np\nimport pandas as pd\nfrom loguru import logger\n\nfrom config.settings import get_settings\nfrom features.rolling_stats import compute_rolling_stats\nfrom features.ewma import compute_ewma, compute_ewma_std, compute_ewma_gradient\nfrom features.fft_features import compute_fft_features\nfrom features.gradient_estimator import (\n    finite_difference_gradient,\n    congestion_onset_score,\n)\n\n\nclass FeaturePipeline:\n    """Orchestrates all feature extraction steps on a telemetry DataFrame.\n\n    The pipeline is **stateless** — ``transform()`` has no side-effects.\n    Column naming convention: ``{feature}_{window}s``\n    (e.g. ``rtt_mean_5s``, ``rtt_p95_30s``, ``ewma_gradient_rtt``).\n\n    Expected input columns:\n        timestamp, rtt_ms, packet_loss, jitter_ms,\n        bytes_per_sec, packets_per_sec\n    """\n\n    def __init__(self) -> None:\n        """Initialise with global settings."""\n        self.settings = get_settings()\n\n    def transform(self, df: pd.DataFrame) -> pd.DataFrame:\n        """Apply all feature extractors and return an enriched DataFrame.\n\n        Args:\n            df: Raw telemetry DataFrame with required columns.\n\n        Returns:\n            Enriched DataFrame with ~45 feature columns plus originals.\n            NaN rows from window edges are dropped.\n        """\n        required_cols = [\n            "timestamp", "rtt_ms", "packet_loss",\n            "jitter_ms", "bytes_per_sec", "packets_per_sec",\n        ]\n        missing = set(required_cols) - set(df.columns)\n        if missing:\n            raise ValueError(f"Missing required columns: {missing}")\n\n        df = df.copy().sort_values("timestamp").reset_index(drop=True)\n        logger.info("FeaturePipeline.transform — input rows: {}", len(df))\n\n        windows = [\n            ("short", self.settings.WINDOW_SHORT),\n            ("medium", self.settings.WINDOW_MEDIUM),\n            ("long", self.settings.WINDOW_LONG),\n        ]\n\n        # ── Rolling statistics for each window ────────────────\n        # Build all new columns into a dict, then concat at the end\n        # to avoid DataFrame fragmentation warnings.\n        new_cols: Dict[str, pd.Series] = {}\n\n        metrics = ["rtt_ms", "jitter_ms", "bytes_per_sec", "packets_per_sec"]\n        for _label, win in windows:\n            win_size = max(1, win)\n            for metric in metrics:\n                prefix = metric.replace("_ms", "").replace("_per_sec", "")\n                stats = compute_rolling_stats(df[metric], window=win_size)\n                for stat_name, stat_series in stats.items():\n                    col_name = f"{prefix}_{stat_name}_{win}s"\n                    new_cols[col_name] = stat_series.values\n\n        # ── EWMA features on rtt_ms ──────────────────────────\n        alpha = self.settings.EWMA_ALPHA\n        new_cols["ewma_rtt"] = compute_ewma(df["rtt_ms"], alpha).values\n        new_cols["ewma_std_rtt"] = compute_ewma_std(df["rtt_ms"], alpha).values\n        new_cols["ewma_gradient_rtt"] = compute_ewma_gradient(df["rtt_ms"], alpha).values\n\n        # ── FFT features over a 256-sample window ─────────────\n        # Compute every 64 rows and forward-fill for performance\n        fft_window = 256\n        fft_step = 64\n        fft_dominant = np.zeros(len(df))\n        fft_entropy = np.zeros(len(df))\n        fft_ratio = np.zeros(len(df))\n\n        if len(df) >= fft_window:\n            for i in range(fft_window, len(df) + 1, fft_step):\n                window_series = df["rtt_ms"].iloc[i - fft_window : i]\n                feats = compute_fft_features(window_series, sample_rate=1.0)\n                end = min(i + fft_step, len(df))\n                fft_dominant[i - 1 : end] = feats["dominant_frequency_hz"]\n                fft_entropy[i - 1 : end] = feats["spectral_entropy"]\n                fft_ratio[i - 1 : end] = feats["spectral_power_ratio"]\n\n        new_cols["fft_dominant_freq"] = fft_dominant\n        new_cols["fft_spectral_entropy"] = fft_entropy\n        new_cols["fft_power_ratio"] = fft_ratio\n\n        # ── Gradient features ─────────────────────────────────\n        new_cols["rtt_gradient_1"] = finite_difference_gradient(df["rtt_ms"], order=1).values\n        new_cols["rtt_gradient_2"] = finite_difference_gradient(df["rtt_ms"], order=2).values\n        onset = congestion_onset_score(df["rtt_ms"], df["bytes_per_sec"])\n        new_cols["congestion_onset_score"] = onset.values\n\n        # ── Packet loss rolling mean ──────────────────────────\n        new_cols["packet_loss_rate_5s"] = (\n            df["packet_loss"].rolling(window=5, min_periods=1).mean().values\n        )\n        new_cols["packet_loss_rate_30s"] = (\n            df["packet_loss"].rolling(window=30, min_periods=1).mean().values\n        )\n\n        # ── Concat all new columns at once to avoid fragmentation ──\n        new_df = pd.DataFrame(new_cols, index=df.index)\n        df = pd.concat([df, new_df], axis=1)\n\n        # ── Drop NaN rows from window edges ───────────────────\n        max_window = max(w for _, w in windows)\n        df = df.iloc[max_window:].reset_index(drop=True)\n\n        logger.info(\n            "FeaturePipeline.transform — output rows: {}, columns: {}",\n            len(df),\n            len(df.columns),\n        )\n\n        return df\n\n',    "models/base_model.py": '"""Abstract base class for all latency prediction models.\n\nEvery model in the system must inherit from ``BaseLatencyModel`` and\nimplement the ``fit``, ``predict``, and ``predict_proba_congestion``\nmethods.\n"""\n\nfrom __future__ import annotations\n\nimport os\nfrom abc import ABC, abstractmethod\nfrom typing import List, Optional\n\nimport joblib\nimport numpy as np\nfrom loguru import logger\n\n\nclass BaseLatencyModel(ABC):\n    """Abstract base for latency / congestion prediction models.\n\n    Subclasses must implement ``fit``, ``predict``, and\n    ``predict_proba_congestion``.  Serialization helpers (``save`` /\n    ``load``) are provided via joblib for sklearn-compatible models;\n    PyTorch models should override as needed.\n    """\n\n    @property\n    @abstractmethod\n    def name(self) -> str:\n        """Human-readable model identifier (e.g. \'arima\', \'xgboost\')."""\n        ...\n\n    @property\n    def feature_names(self) -> List[str]:\n        """Return the list of feature names the model was trained on.\n\n        Override in subclasses that track feature names.\n        """\n        return getattr(self, "_feature_names", [])\n\n    @abstractmethod\n    def fit(self, X: np.ndarray, y: np.ndarray) -> None:\n        """Train the model on feature matrix X and targets y.\n\n        Args:\n            X: Feature matrix of shape (n_samples, n_features).\n            y: Target array (shape depends on model).\n        """\n        ...\n\n    @abstractmethod\n    def predict(self, X: np.ndarray) -> np.ndarray:\n        """Predict latency for given features.\n\n        Args:\n            X: Feature matrix (n_samples, n_features).\n\n        Returns:\n            Array of shape (n_samples, 3) with predictions for\n            t+10 s, t+30 s, t+60 s horizons.\n        """\n        ...\n\n    @abstractmethod\n    def predict_proba_congestion(self, X: np.ndarray) -> np.ndarray:\n        """Predict congestion probability.\n\n        Args:\n            X: Feature matrix.\n\n        Returns:\n            Array of shape (n_samples,) with probabilities in [0, 1].\n        """\n        ...\n\n    def save(self, path: str) -> None:\n        """Serialize the model to disk.\n\n        Args:\n            path: File path (directory will be created if needed).\n        """\n        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)\n        joblib.dump(self, path)\n        logger.info("Model \'{}\' saved to {}", self.name, path)\n\n    @classmethod\n    def load(cls, path: str) -> "BaseLatencyModel":\n        """Deserialize a model from disk.\n\n        Args:\n            path: File path to the saved model.\n\n        Returns:\n            Loaded model instance.\n\n        Raises:\n            FileNotFoundError: If path does not exist.\n        """\n        if not os.path.exists(path):\n            raise FileNotFoundError(f"Model file not found: {path}")\n        model = joblib.load(path)\n        logger.info("Model \'{}\' loaded from {}", model.name, path)\n        return model\n',    "models/arima_model.py": '"""ARIMA baseline model for univariate RTT forecasting.\n\nUses ``statsmodels.SARIMAX`` with AIC-based grid search for (p, d, q)\norder selection.\n"""\n\nfrom __future__ import annotations\n\nimport warnings\nfrom itertools import product\nfrom typing import List, Optional, Tuple\n\nimport numpy as np\nfrom loguru import logger\n\nfrom models.base_model import BaseLatencyModel\n\ntry:\n    from statsmodels.tsa.statespace.sarimax import SARIMAX\n    STATSMODELS_AVAILABLE = True\nexcept ImportError:\n    STATSMODELS_AVAILABLE = False\n\n\nclass ARIMAModel(BaseLatencyModel):\n    """Univariate ARIMA model fitted on the rtt_ms time series.\n\n    Selects optimal (p, d, q) via AIC minimisation over a small grid.\n    Forecasts RTT at t+10 s, t+30 s, t+60 s horizons.\n    """\n\n    def __init__(self) -> None:\n        self._model = None\n        self._fitted = None\n        self._order: Tuple[int, int, int] = (1, 1, 1)\n        self._baseline_mean: float = 0.0\n        self._baseline_std: float = 1.0\n        self._feature_names: List[str] = ["rtt_ms"]\n\n    @property\n    def name(self) -> str:\n        return "arima"\n\n    def fit(self, X: np.ndarray, y: np.ndarray) -> None:\n        """Fit ARIMA on the RTT series (y) using AIC grid search.\n\n        Args:\n            X: Ignored (ARIMA is univariate).\n            y: 1-D array of rtt_ms values.\n        """\n        if not STATSMODELS_AVAILABLE:\n            raise RuntimeError("statsmodels is required for ARIMAModel")\n\n        series = np.asarray(y).flatten()\n        self._baseline_mean = float(np.mean(series))\n        self._baseline_std = float(np.std(series))\n\n        best_aic = float("inf")\n        best_order = (1, 1, 1)\n\n        p_range = [0, 1, 2]\n        d_range = [0, 1]\n        q_range = [0, 1, 2]\n\n        for p, d, q in product(p_range, d_range, q_range):\n            try:\n                with warnings.catch_warnings():\n                    warnings.simplefilter("ignore")\n                    model = SARIMAX(series, order=(p, d, q), enforce_stationarity=False,\n                                    enforce_invertibility=False)\n                    fitted = model.fit(disp=False, maxiter=100)\n                    if fitted.aic < best_aic:\n                        best_aic = fitted.aic\n                        best_order = (p, d, q)\n            except (ValueError, np.linalg.LinAlgError, RuntimeError):\n                continue\n\n        self._order = best_order\n        logger.info("ARIMA best order: {} with AIC={:.2f}", best_order, best_aic)\n\n        with warnings.catch_warnings():\n            warnings.simplefilter("ignore")\n            self._model = SARIMAX(series, order=best_order,\n                                  enforce_stationarity=False,\n                                  enforce_invertibility=False)\n            self._fitted = self._model.fit(disp=False, maxiter=200)\n\n    def predict(self, X: np.ndarray) -> np.ndarray:\n        """Forecast RTT at 10 s, 30 s, and 60 s ahead.\n\n        Args:\n            X: Ignored (uses internal state).\n\n        Returns:\n            Array of shape (1, 3) with [rtt_10s, rtt_30s, rtt_60s].\n        """\n        if self._fitted is None:\n            raise RuntimeError("Model not fitted — call fit() first")\n\n        with warnings.catch_warnings():\n            warnings.simplefilter("ignore")\n            forecast = self._fitted.forecast(steps=60)\n\n        rtt_10 = float(forecast[9]) if len(forecast) > 9 else float(forecast[-1])\n        rtt_30 = float(forecast[29]) if len(forecast) > 29 else float(forecast[-1])\n        rtt_60 = float(forecast[59]) if len(forecast) > 59 else float(forecast[-1])\n\n        return np.array([[max(0.0, rtt_10), max(0.0, rtt_30), max(0.0, rtt_60)]])\n\n    def predict_proba_congestion(self, X: np.ndarray) -> np.ndarray:\n        """Return congestion probability based on forecast vs. baseline.\n\n        Returns 1.0 if forecast > baseline_mean + 2 * baseline_std.\n\n        Args:\n            X: Ignored.\n\n        Returns:\n            Array of shape (1,) with congestion probability.\n        """\n        if self._fitted is None:\n            return np.array([0.0])\n\n        preds = self.predict(X)\n        avg_pred = float(np.mean(preds))\n        threshold = self._baseline_mean + 2 * self._baseline_std\n\n        if avg_pred > threshold:\n            return np.array([1.0])\n        elif avg_pred > self._baseline_mean + self._baseline_std:\n            ratio = (avg_pred - self._baseline_mean) / (2 * self._baseline_std + 1e-9)\n            return np.array([min(1.0, max(0.0, ratio))])\n        else:\n            return np.array([0.0])\n',    "models/xgboost_model.py": '"""XGBoost gradient-boosted tree model for latency and congestion prediction.\n\nUses three sub-models:\n  1. XGBRegressor for RTT prediction\n  2. XGBRegressor for jitter prediction\n  3. XGBClassifier for congestion probability\nSupports SHAP-based explainability.\n"""\n\nfrom __future__ import annotations\n\nimport os\nfrom typing import Any, Dict, List, Optional\n\nimport joblib\nimport numpy as np\nfrom loguru import logger\n\nfrom models.base_model import BaseLatencyModel\n\ntry:\n    from xgboost import XGBClassifier, XGBRegressor\n    XGBOOST_AVAILABLE = True\nexcept ImportError:\n    XGBOOST_AVAILABLE = False\n    logger.warning("xgboost not installed — XGBoostModel unavailable")\n\n\nclass XGBoostModel(BaseLatencyModel):\n    """Gradient-boosted model for multi-horizon latency prediction.\n\n    Trains separate regressors for each RTT horizon (10 s, 30 s, 60 s)\n    and a single classifier for congestion probability.\n\n    Attributes:\n        regressors: Dict mapping horizon label to XGBRegressor.\n        classifier: XGBClassifier for congestion labels.\n    """\n\n    def __init__(self) -> None:\n        self.regressors: Dict[str, Any] = {}\n        self.jitter_regressor: Optional[Any] = None\n        self.classifier: Optional[Any] = None\n        self._feature_names: List[str] = []\n        self._feature_importances: Dict[str, np.ndarray] = {}\n\n        self._xgb_params = dict(\n            n_estimators=500,\n            max_depth=6,\n            learning_rate=0.05,\n            subsample=0.8,\n            colsample_bytree=0.8,\n            random_state=42,\n            n_jobs=-1,\n        )\n\n    @property\n    def name(self) -> str:\n        return "xgboost"\n\n    def fit(self, X: np.ndarray, y: np.ndarray, **kwargs: Any) -> None:\n        """Train regressors and classifier.\n\n        Args:\n            X: Feature matrix (n_samples, n_features).\n            y: Target matrix with columns:\n               [rtt_10s, rtt_30s, rtt_60s, jitter, congestion_label].\n               If y has fewer columns, only available targets are used.\n            **kwargs: Optional ``eval_set`` for early stopping.\n        """\n        if not XGBOOST_AVAILABLE:\n            raise RuntimeError("xgboost is required for XGBoostModel")\n\n        self._feature_names = kwargs.get("feature_names", [])\n\n        eval_set = kwargs.get("eval_set")\n        fit_params: Dict[str, Any] = {}\n        if eval_set is not None:\n            fit_params["eval_set"] = eval_set\n            fit_params["verbose"] = False\n\n        horizons = ["rtt_10s", "rtt_30s", "rtt_60s"]\n        for i, hz in enumerate(horizons):\n            if y.shape[1] <= i:\n                break\n            logger.info("Training XGBRegressor for {}", hz)\n            reg = XGBRegressor(**self._xgb_params)\n\n            if eval_set is not None:\n                X_val, y_val = eval_set[0]\n                reg.fit(\n                    X, y[:, i],\n                    eval_set=[(X_val, y_val[:, i])],\n                    verbose=False,\n                )\n            else:\n                reg.fit(X, y[:, i])\n\n            self.regressors[hz] = reg\n            self._feature_importances[hz] = reg.feature_importances_\n\n        # Jitter regressor\n        if y.shape[1] > 3:\n            logger.info("Training XGBRegressor for jitter")\n            self.jitter_regressor = XGBRegressor(**self._xgb_params)\n            if eval_set is not None:\n                X_val, y_val = eval_set[0]\n                self.jitter_regressor.fit(\n                    X, y[:, 3],\n                    eval_set=[(X_val, y_val[:, 3])],\n                    verbose=False,\n                )\n            else:\n                self.jitter_regressor.fit(X, y[:, 3])\n\n        # Congestion classifier\n        if y.shape[1] > 4:\n            logger.info("Training XGBClassifier for congestion")\n            self.classifier = XGBClassifier(\n                **self._xgb_params,\n                objective="binary:logistic",\n                eval_metric="logloss",\n            )\n            if eval_set is not None:\n                X_val, y_val = eval_set[0]\n                self.classifier.fit(\n                    X, y[:, 4].astype(int),\n                    eval_set=[(X_val, y_val[:, 4].astype(int))],\n                    verbose=False,\n                )\n            else:\n                self.classifier.fit(X, y[:, 4].astype(int))\n\n    def predict(self, X: np.ndarray) -> np.ndarray:\n        """Predict RTT at 10 s, 30 s, and 60 s horizons.\n\n        Args:\n            X: Feature matrix (n_samples, n_features).\n\n        Returns:\n            Array of shape (n_samples, 3).\n        """\n        results = []\n        for hz in ["rtt_10s", "rtt_30s", "rtt_60s"]:\n            if hz in self.regressors:\n                pred = self.regressors[hz].predict(X)\n                results.append(np.clip(pred, 0, None))\n            else:\n                results.append(np.full(X.shape[0], 20.0))\n\n        return np.column_stack(results)\n\n    def predict_proba_congestion(self, X: np.ndarray) -> np.ndarray:\n        """Predict congestion probability.\n\n        Args:\n            X: Feature matrix.\n\n        Returns:\n            Array of shape (n_samples,) with probabilities in [0, 1].\n        """\n        if self.classifier is None:\n            return np.zeros(X.shape[0])\n\n        proba = self.classifier.predict_proba(X)\n        # Column 1 = probability of class 1 (congestion)\n        if proba.shape[1] == 2:\n            return proba[:, 1]\n        return proba[:, 0]\n\n    def explain(self, X: np.ndarray) -> Dict[str, Any]:\n        """Compute SHAP values for explainability.\n\n        Args:\n            X: Feature matrix.\n\n        Returns:\n            Dictionary mapping horizon to SHAP value arrays.\n        """\n        explanations: Dict[str, Any] = {}\n        try:\n            import shap\n            for hz, reg in self.regressors.items():\n                explainer = shap.TreeExplainer(reg)\n                shap_values = explainer.shap_values(X)\n                explanations[hz] = shap_values\n        except ImportError:\n            logger.warning("shap not installed — explain() unavailable")\n        except Exception as exc:\n            logger.error("SHAP explanation failed: {}", exc)\n\n        return explanations\n\n    def save(self, path: str) -> None:\n        """Save all sub-models to disk.\n\n        Args:\n            path: Base path (without extension).\n        """\n        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)\n        joblib.dump(\n            {\n                "regressors": self.regressors,\n                "jitter_regressor": self.jitter_regressor,\n                "classifier": self.classifier,\n                "feature_names": self._feature_names,\n                "feature_importances": self._feature_importances,\n                "xgb_params": self._xgb_params,\n            },\n            path,\n        )\n        logger.info("XGBoostModel saved to {}", path)\n\n    @classmethod\n    def load(cls, path: str) -> "XGBoostModel":\n        """Load a saved XGBoostModel.\n\n        Args:\n            path: Path to the saved model file.\n\n        Returns:\n            Restored XGBoostModel instance.\n        """\n        if not os.path.exists(path):\n            raise FileNotFoundError(f"Model file not found: {path}")\n\n        data = joblib.load(path)\n        model = cls()\n        model.regressors = data["regressors"]\n        model.jitter_regressor = data["jitter_regressor"]\n        model.classifier = data["classifier"]\n        model._feature_names = data.get("feature_names", [])\n        model._feature_importances = data.get("feature_importances", {})\n        model._xgb_params = data.get("xgb_params", model._xgb_params)\n        logger.info("XGBoostModel loaded from {}", path)\n        return model\n',    "models/lstm_model.py": '"""PyTorch LSTM model with self-attention for latency prediction.\n\nArchitecture:\n  - Input: (batch, seq_len=60, n_features≈45)\n  - LSTM: 2 layers, hidden=128, dropout=0.2\n  - Self-attention: single-head over LSTM outputs\n  - Two output heads:\n      latency_head  → (batch, 3) for t+10/30/60 s RTT\n      congestion_head → (batch, 1) sigmoid for congestion prob\n"""\n\nfrom __future__ import annotations\n\nimport os\nfrom typing import Any, Dict, List, Optional, Tuple\n\nimport numpy as np\nfrom loguru import logger\n\nfrom models.base_model import BaseLatencyModel\n\ntry:\n    import torch\n    import torch.nn as nn\n    from torch.utils.data import DataLoader, TensorDataset\n    TORCH_AVAILABLE = True\nexcept ImportError:\n    TORCH_AVAILABLE = False\n    logger.warning("PyTorch not installed — LSTMModel unavailable")\n\ntry:\n    from sklearn.preprocessing import StandardScaler\nexcept ImportError:\n    StandardScaler = None  # type: ignore[assignment,misc]\n\n\n# ──────────────────────────────────────────────────────────────\n# PyTorch module definition\n# ──────────────────────────────────────────────────────────────\n\nif TORCH_AVAILABLE:\n\n    class _LSTMNetwork(nn.Module):\n        """LSTM + self-attention network for latency and congestion."""\n\n        def __init__(self, n_features: int, hidden_size: int = 128,\n                     n_layers: int = 2, dropout: float = 0.2) -> None:\n            super().__init__()\n            self.lstm = nn.LSTM(\n                input_size=n_features,\n                hidden_size=hidden_size,\n                num_layers=n_layers,\n                dropout=dropout,\n                batch_first=True,\n            )\n            # Single-head self-attention\n            self.attention_weight = nn.Linear(hidden_size, 1)\n\n            # Output heads\n            self.latency_head = nn.Linear(hidden_size, 3)\n            self.congestion_head = nn.Sequential(\n                nn.Linear(hidden_size, 1),\n                nn.Sigmoid(),\n            )\n\n        def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:\n            """Forward pass.\n\n            Args:\n                x: (batch, seq_len, n_features).\n\n            Returns:\n                Tuple of (latency_preds, congestion_probs) with shapes\n                (batch, 3) and (batch, 1).\n            """\n            lstm_out, _ = self.lstm(x)  # (batch, seq, hidden)\n\n            # Attention weights\n            attn_scores = self.attention_weight(lstm_out).squeeze(-1)  # (batch, seq)\n            attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)  # (batch, seq, 1)\n            context = (lstm_out * attn_weights).sum(dim=1)  # (batch, hidden)\n\n            latency = self.latency_head(context)  # (batch, 3)\n            congestion = self.congestion_head(context)  # (batch, 1)\n\n            return latency, congestion\n\n\nclass LSTMModel(BaseLatencyModel):\n    """LSTM model with attention for multi-horizon latency prediction.\n\n    Normalises features with ``StandardScaler`` (fit on training data\n    only) and stores the scaler alongside the model checkpoint.\n    """\n\n    SEQ_LEN = 60\n\n    def __init__(self, n_features: int = 45, hidden_size: int = 128,\n                 n_layers: int = 2, dropout: float = 0.2) -> None:\n        self.n_features = n_features\n        self.hidden_size = hidden_size\n        self.n_layers = n_layers\n        self.dropout = dropout\n        self._model: Optional[Any] = None\n        self._scaler: Optional[Any] = None\n        self._target_scaler: Optional[Any] = None\n        self._feature_names: List[str] = []\n        self._device = "cpu"\n\n        if TORCH_AVAILABLE and torch.cuda.is_available():\n            self._device = "cuda"\n        elif TORCH_AVAILABLE and hasattr(torch.backends, "mps") and torch.backends.mps.is_available():\n            self._device = "mps"\n\n    @property\n    def name(self) -> str:\n        return "lstm"\n\n    def _create_sequences(\n        self, X: np.ndarray, y: Optional[np.ndarray] = None, step: int = 1\n    ) -> Tuple[np.ndarray, Optional[np.ndarray], Optional[np.ndarray]]:\n        """Create sliding-window sequences for the LSTM.\n\n        Args:\n            X: Feature matrix (n_samples, n_features).\n            y: Targets (n_samples, 3+1): rtt horizons + congestion.\n            step: Stride between windows.\n\n        Returns:\n            Tuple of (X_seq, y_latency, y_congestion) or\n            (X_seq, None, None) if y is None.\n        """\n        sequences = []\n        targets_lat = [] if y is not None else None\n        targets_cong = [] if y is not None else None\n\n        for i in range(0, len(X) - self.SEQ_LEN + 1, step):\n            sequences.append(X[i : i + self.SEQ_LEN])\n            if y is not None:\n                idx = i + self.SEQ_LEN - 1\n                targets_lat.append(y[idx, :3])\n                targets_cong.append(y[idx, 3] if y.shape[1] > 3 else 0.0)\n\n        X_seq = np.array(sequences, dtype=np.float32)\n        y_lat = np.array(targets_lat, dtype=np.float32) if targets_lat else None\n        y_cong = np.array(targets_cong, dtype=np.float32) if targets_cong else None\n\n        return X_seq, y_lat, y_cong\n\n    def fit(self, X: np.ndarray, y: np.ndarray, **kwargs: Any) -> None:\n        """Train the LSTM model.\n\n        Args:\n            X: Feature matrix.\n            y: Target matrix [rtt_10s, rtt_30s, rtt_60s, congestion].\n            **kwargs: Optional X_val, y_val for early stopping.\n        """\n        if not TORCH_AVAILABLE:\n            raise RuntimeError("PyTorch is required for LSTMModel")\n\n        # Fit scaler on training features\n        self._scaler = StandardScaler()\n        X_scaled = self._scaler.fit_transform(X)\n\n        self.n_features = X.shape[1]\n        self._model = _LSTMNetwork(\n            n_features=self.n_features,\n            hidden_size=self.hidden_size,\n            n_layers=self.n_layers,\n            dropout=self.dropout,\n        ).to(self._device)\n\n        # Create sequences — use step>1 to reduce memory on large datasets.\n        # step=5 gives 5× fewer sequences, cutting RAM from ~600 MB to ~120 MB.\n        seq_step = kwargs.get("seq_step", max(1, len(X) // 20_000))\n        X_seq, y_lat, y_cong = self._create_sequences(X_scaled, y, step=seq_step)\n        del X_scaled  # free memory\n        if X_seq.shape[0] == 0:\n            logger.warning("Not enough data for LSTM sequences")\n            return\n\n        logger.info("LSTM sequences: {} (step={})", X_seq.shape[0], seq_step)\n\n        dataset = TensorDataset(\n            torch.tensor(X_seq, dtype=torch.float32),\n            torch.tensor(y_lat, dtype=torch.float32),\n            torch.tensor(y_cong, dtype=torch.float32),\n        )\n        del X_seq, y_lat, y_cong  # free numpy arrays, data lives in tensors now\n        loader = DataLoader(dataset, batch_size=64, shuffle=True)\n\n        # Validation set\n        val_loader = None\n        X_val = kwargs.get("X_val")\n        y_val = kwargs.get("y_val")\n        if X_val is not None and y_val is not None:\n            X_val_scaled = self._scaler.transform(X_val)\n            Xv_seq, yv_lat, yv_cong = self._create_sequences(X_val_scaled, y_val, step=seq_step)\n            del X_val_scaled\n            if Xv_seq.shape[0] > 0:\n                val_ds = TensorDataset(\n                    torch.tensor(Xv_seq, dtype=torch.float32),\n                    torch.tensor(yv_lat, dtype=torch.float32),\n                    torch.tensor(yv_cong, dtype=torch.float32),\n                )\n                del Xv_seq, yv_lat, yv_cong\n                val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)\n\n        # Training loop\n        optimizer = torch.optim.AdamW(\n            self._model.parameters(), lr=1e-3, weight_decay=1e-4\n        )\n        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)\n        mse_loss = nn.MSELoss()\n        bce_loss = nn.BCELoss()\n\n        best_val_loss = float("inf")\n        patience_counter = 0\n        patience = 10\n        epochs = 50\n\n        for epoch in range(epochs):\n            self._model.train()\n            train_loss = 0.0\n            for X_batch, y_lat_batch, y_cong_batch in loader:\n                X_batch = X_batch.to(self._device)\n                y_lat_batch = y_lat_batch.to(self._device)\n                y_cong_batch = y_cong_batch.to(self._device)\n\n                optimizer.zero_grad()\n                lat_pred, cong_pred = self._model(X_batch)\n\n                loss_lat = mse_loss(lat_pred, y_lat_batch)\n                loss_cong = bce_loss(cong_pred.squeeze(), y_cong_batch)\n                loss = loss_lat + 0.3 * loss_cong\n\n                loss.backward()\n                torch.nn.utils.clip_grad_norm_(self._model.parameters(), max_norm=1.0)\n                optimizer.step()\n                train_loss += loss.item()\n\n            scheduler.step()\n\n            # Validation\n            if val_loader is not None:\n                self._model.eval()\n                val_loss = 0.0\n                with torch.no_grad():\n                    for X_b, yl_b, yc_b in val_loader:\n                        X_b = X_b.to(self._device)\n                        yl_b = yl_b.to(self._device)\n                        yc_b = yc_b.to(self._device)\n                        lp, cp = self._model(X_b)\n                        vl = mse_loss(lp, yl_b) + 0.3 * bce_loss(cp.squeeze(), yc_b)\n                        val_loss += vl.item()\n\n                if val_loss < best_val_loss:\n                    best_val_loss = val_loss\n                    patience_counter = 0\n                else:\n                    patience_counter += 1\n                    if patience_counter >= patience:\n                        logger.info("Early stopping at epoch {}", epoch + 1)\n                        break\n\n            if (epoch + 1) % 10 == 0:\n                logger.info(\n                    "Epoch {}/{} — train_loss={:.4f}",\n                    epoch + 1, epochs, train_loss / len(loader),\n                )\n\n    def predict(self, X: np.ndarray) -> np.ndarray:\n        """Predict RTT at three horizons.\n\n        Args:\n            X: Feature matrix (>= SEQ_LEN rows).\n\n        Returns:\n            Array of shape (n_pred, 3).\n        """\n        if self._model is None or self._scaler is None:\n            raise RuntimeError("Model not fitted")\n\n        self._model.eval()\n        X_scaled = self._scaler.transform(X)\n        X_seq, _, _ = self._create_sequences(X_scaled)\n\n        if X_seq.shape[0] == 0:\n            # Not enough data for a full sequence — use last SEQ_LEN rows\n            if len(X_scaled) >= self.SEQ_LEN:\n                X_seq = X_scaled[-self.SEQ_LEN:][np.newaxis, ...]\n            else:\n                return np.full((1, 3), 20.0)\n\n        with torch.no_grad():\n            X_t = torch.tensor(X_seq, dtype=torch.float32).to(self._device)\n            lat_pred, _ = self._model(X_t)\n            result = lat_pred.cpu().numpy()\n\n        return np.clip(result, 0, None)\n\n    def predict_proba_congestion(self, X: np.ndarray) -> np.ndarray:\n        """Predict congestion probability.\n\n        Args:\n            X: Feature matrix.\n\n        Returns:\n            Array of shape (n_pred,) with probabilities in [0, 1].\n        """\n        if self._model is None or self._scaler is None:\n            return np.zeros(1)\n\n        self._model.eval()\n        X_scaled = self._scaler.transform(X)\n        X_seq, _, _ = self._create_sequences(X_scaled)\n\n        if X_seq.shape[0] == 0:\n            if len(X_scaled) >= self.SEQ_LEN:\n                X_seq = X_scaled[-self.SEQ_LEN:][np.newaxis, ...]\n            else:\n                return np.array([0.0])\n\n        with torch.no_grad():\n            X_t = torch.tensor(X_seq, dtype=torch.float32).to(self._device)\n            _, cong_pred = self._model(X_t)\n            return cong_pred.squeeze(-1).cpu().numpy()\n\n    def save(self, path: str) -> None:\n        """Save model weights, scaler, and config.\n\n        Args:\n            path: File path for the checkpoint.\n        """\n        if not TORCH_AVAILABLE or self._model is None:\n            logger.warning("No model to save")\n            return\n\n        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)\n        checkpoint = {\n            "model_state": self._model.state_dict(),\n            "scaler": self._scaler,\n            "n_features": self.n_features,\n            "hidden_size": self.hidden_size,\n            "n_layers": self.n_layers,\n            "dropout": self.dropout,\n            "feature_names": self._feature_names,\n        }\n        torch.save(checkpoint, path)\n        logger.info("LSTMModel saved to {}", path)\n\n    @classmethod\n    def load(cls, path: str) -> "LSTMModel":\n        """Load a saved LSTMModel checkpoint.\n\n        Args:\n            path: Path to the checkpoint file.\n\n        Returns:\n            Restored LSTMModel instance.\n        """\n        if not TORCH_AVAILABLE:\n            raise RuntimeError("PyTorch required to load LSTMModel")\n\n        if not os.path.exists(path):\n            raise FileNotFoundError(f"Checkpoint not found: {path}")\n\n        checkpoint = torch.load(path, map_location="cpu", weights_only=False)\n        model = cls(\n            n_features=checkpoint["n_features"],\n            hidden_size=checkpoint["hidden_size"],\n            n_layers=checkpoint["n_layers"],\n            dropout=checkpoint["dropout"],\n        )\n        model._scaler = checkpoint["scaler"]\n        model._feature_names = checkpoint.get("feature_names", [])\n        model._model = _LSTMNetwork(\n            n_features=model.n_features,\n            hidden_size=model.hidden_size,\n            n_layers=model.n_layers,\n            dropout=model.dropout,\n        ).to(model._device)\n        model._model.load_state_dict(checkpoint["model_state"])\n        logger.info("LSTMModel loaded from {}", path)\n        return model\n',    "models/queuing_model.py": '"""M/M/1 and M/D/1 queuing-theory analytical models for latency prediction.\n\nAutomatically selects M/M/1 or M/D/1 based on the coefficient of\nvariation of inter-arrival times.\n"""\n\nfrom __future__ import annotations\n\nfrom typing import List\n\nimport numpy as np\nfrom loguru import logger\n\nfrom models.base_model import BaseLatencyModel\n\n\nclass QueuingModel(BaseLatencyModel):\n    """Analytical queuing model for latency estimation.\n\n    M/M/1: latency = 1 / (μ − λ)\n    M/D/1: latency = (1 / (μ − λ)) × (1 − λ / (2μ))\n\n    Selection is automatic: if CV of inter-arrival times < 0.5 → M/D/1,\n    otherwise M/M/1.\n\n    Attributes:\n        mu: Estimated service rate (packets/sec) derived from training.\n        model_type: \'mm1\' or \'md1\'.\n    """\n\n    def __init__(self) -> None:\n        self.mu: float = 1000.0  # default service rate\n        self.model_type: str = "mm1"\n        self._baseline_rtt: float = 20.0\n        self._feature_names: List[str] = ["packets_per_sec", "jitter_ms"]\n\n    @property\n    def name(self) -> str:\n        return "queuing"\n\n    def fit(self, X: np.ndarray, y: np.ndarray) -> None:\n        """Estimate service rate μ from training data.\n\n        μ is derived from the minimum observed RTT and average\n        packet rate:  μ ≈ avg_pps × (baseline_rtt / min_rtt).\n\n        Also determines whether to use M/M/1 or M/D/1 based on\n        inter-arrival time variation.\n\n        Args:\n            X: Feature matrix — expects at least columns for\n               packets_per_sec (col 0) and jitter_ms (col 1).\n            y: Target rtt_ms values (1-D array).\n        """\n        rtt = y.flatten() if y.ndim > 1 else y\n        self._baseline_rtt = float(np.median(rtt))\n        min_rtt = float(np.min(rtt[rtt > 0])) if np.any(rtt > 0) else 1.0\n\n        # Extract packets_per_sec (first column)\n        if X.ndim == 2 and X.shape[1] >= 1:\n            pps = X[:, 0]\n        else:\n            pps = np.full(len(rtt), 1000.0)\n\n        avg_pps = float(np.mean(pps))\n        self.mu = avg_pps * (self._baseline_rtt / min_rtt) * 1.2  # headroom\n\n        # Determine model type from CV of inter-arrival proxy (jitter)\n        if X.ndim == 2 and X.shape[1] >= 2:\n            jitter = X[:, 1]\n            cv = float(np.std(jitter) / (np.mean(jitter) + 1e-9))\n            self.model_type = "md1" if cv < 0.5 else "mm1"\n        else:\n            self.model_type = "mm1"\n\n        logger.info(\n            "QueuingModel fitted: mu={:.1f}, type={}, baseline_rtt={:.1f}ms",\n            self.mu, self.model_type, self._baseline_rtt,\n        )\n\n    def _compute_latency(self, lam: float) -> float:\n        """Compute queuing latency for arrival rate λ.\n\n        Args:\n            lam: Arrival rate (packets/sec).\n\n        Returns:\n            Estimated latency in milliseconds.\n        """\n        if lam >= self.mu:\n            # System at or over capacity — return high latency\n            return self._baseline_rtt * 10.0\n\n        if self.model_type == "md1":\n            latency_sec = (1.0 / (self.mu - lam)) * (1.0 - lam / (2.0 * self.mu))\n        else:\n            latency_sec = 1.0 / (self.mu - lam)\n\n        latency_ms = latency_sec * 1000.0\n        return max(latency_ms, 0.1)\n\n    def predict(self, X: np.ndarray) -> np.ndarray:\n        """Predict RTT for t+10 s, t+30 s, t+60 s.\n\n        The queuing model returns the same value for all horizons\n        (steady-state assumption) but scales by a growth factor if\n        utilization is trending up.\n\n        Args:\n            X: Feature matrix with packets_per_sec as first column.\n\n        Returns:\n            Array of shape (n_samples, 3).\n        """\n        if X.ndim == 1:\n            X = X.reshape(1, -1)\n\n        results = []\n        for i in range(X.shape[0]):\n            lam = float(X[i, 0]) if X.shape[1] >= 1 else 500.0\n            base_lat = self._compute_latency(lam)\n\n            # Slight scaling for longer horizons (assumption: load may grow)\n            rtt_10 = base_lat\n            rtt_30 = base_lat * 1.05\n            rtt_60 = base_lat * 1.10\n            results.append([rtt_10, rtt_30, rtt_60])\n\n        return np.array(results)\n\n    def predict_proba_congestion(self, X: np.ndarray) -> np.ndarray:\n        """Return utilization ratio λ/μ as congestion probability.\n\n        Args:\n            X: Feature matrix with packets_per_sec as first column.\n\n        Returns:\n            Array of shape (n_samples,) clipped to [0, 1].\n        """\n        if X.ndim == 1:\n            X = X.reshape(1, -1)\n\n        probs = []\n        for i in range(X.shape[0]):\n            lam = float(X[i, 0]) if X.shape[1] >= 1 else 500.0\n            rho = lam / self.mu if self.mu > 0 else 1.0\n            probs.append(min(max(rho, 0.0), 1.0))\n\n        return np.array(probs)\n',    "models/ensemble.py": '"""Weighted ensemble of all prediction models.\n\nCombines ARIMA, XGBoost, LSTM, and queuing model predictions via\nconfigurable weights.  Includes dynamic weight adjustment — if a model\'s\nrecent MAE exceeds 2× the average MAE, its weight is halved and\nredistributed.\n"""\n\nfrom __future__ import annotations\n\nimport os\nfrom typing import Any, Dict, List, Optional\n\nimport numpy as np\nfrom loguru import logger\n\nfrom config.settings import get_settings\nfrom models.base_model import BaseLatencyModel\n\n\nclass EnsembleModel:\n    """Weighted ensemble combining all individual models.\n\n    Attributes:\n        models: Dictionary mapping model name to model instance.\n        weights: Dictionary mapping model name to weight (must sum to 1).\n    """\n\n    def __init__(self) -> None:\n        self.settings = get_settings()\n        self.models: Dict[str, BaseLatencyModel] = {}\n        self.weights: Dict[str, float] = dict(self.settings.ENSEMBLE_WEIGHTS)\n        self._recent_errors: Dict[str, List[float]] = {}\n\n    def add_model(self, model: BaseLatencyModel) -> None:\n        """Register a model in the ensemble.\n\n        Args:\n            model: An instance of BaseLatencyModel.\n        """\n        self.models[model.name] = model\n        if model.name not in self.weights:\n            self.weights[model.name] = 0.1\n        self._recent_errors[model.name] = []\n        logger.info("Ensemble: added model \'{}\'", model.name)\n\n    def _normalize_weights(self) -> Dict[str, float]:\n        """Normalize weights to sum to 1.0 for available models.\n\n        Returns:\n            Normalized weight dict.\n        """\n        available = {k: v for k, v in self.weights.items() if k in self.models}\n        total = sum(available.values())\n        if total <= 0:\n            n = len(available)\n            return {k: 1.0 / n for k in available} if n > 0 else {}\n        return {k: v / total for k, v in available.items()}\n\n    def predict(self, X: np.ndarray) -> Dict[str, Any]:\n        """Generate a weighted ensemble prediction.\n\n        If an individual model fails, it is skipped and its weight\n        redistributed.\n\n        Args:\n            X: Feature matrix.\n\n        Returns:\n            Dictionary with keys: rtt_10s, rtt_30s, rtt_60s,\n            congestion_prob, congestion_label, model_contributions.\n        """\n        weights = self._normalize_weights()\n        predictions: Dict[str, np.ndarray] = {}\n        congestion_probs: Dict[str, float] = {}\n        failed_models: List[str] = []\n\n        for model_name, model in self.models.items():\n            try:\n                pred = model.predict(X)\n                if pred.ndim == 1:\n                    pred = pred.reshape(1, -1)\n                # Use last prediction row\n                predictions[model_name] = pred[-1]\n\n                cong = model.predict_proba_congestion(X)\n                congestion_probs[model_name] = float(cong[-1] if len(cong) > 0 else 0.0)\n            except Exception as exc:\n                logger.error("Model \'{}\' prediction failed: {}", model_name, exc)\n                failed_models.append(model_name)\n\n        if not predictions:\n            logger.error("All models failed — returning defaults")\n            return {\n                "rtt_10s": 20.0,\n                "rtt_30s": 20.0,\n                "rtt_60s": 20.0,\n                "congestion_prob": 0.0,\n                "congestion_label": False,\n                "model_contributions": {},\n            }\n\n        # Recalculate weights excluding failed models\n        active_weights = {k: v for k, v in weights.items() if k not in failed_models and k in predictions}\n        total_w = sum(active_weights.values())\n        if total_w > 0:\n            active_weights = {k: v / total_w for k, v in active_weights.items()}\n        else:\n            n = len(predictions)\n            active_weights = {k: 1.0 / n for k in predictions}\n\n        # Weighted average\n        rtt_ensemble = np.zeros(3)\n        cong_ensemble = 0.0\n        contributions: Dict[str, Dict[str, float]] = {}\n\n        for model_name, pred in predictions.items():\n            w = active_weights.get(model_name, 0.0)\n            weighted_pred = pred[:3] * w\n            rtt_ensemble += weighted_pred\n\n            cong_w = congestion_probs.get(model_name, 0.0) * w\n            cong_ensemble += cong_w\n\n            contributions[model_name] = {\n                "weight": round(w, 4),\n                "rtt_10s": round(float(pred[0]), 3) if len(pred) > 0 else 0.0,\n                "rtt_30s": round(float(pred[1]), 3) if len(pred) > 1 else 0.0,\n                "rtt_60s": round(float(pred[2]), 3) if len(pred) > 2 else 0.0,\n                "congestion_prob": round(congestion_probs.get(model_name, 0.0), 4),\n            }\n\n        return {\n            "rtt_10s": round(float(rtt_ensemble[0]), 3),\n            "rtt_30s": round(float(rtt_ensemble[1]), 3),\n            "rtt_60s": round(float(rtt_ensemble[2]), 3),\n            "congestion_prob": round(float(min(cong_ensemble, 1.0)), 4),\n            "congestion_label": cong_ensemble > 0.7,\n            "model_contributions": contributions,\n        }\n\n    def update_errors(self, model_name: str, mae: float) -> None:\n        """Record recent MAE for dynamic weight adjustment.\n\n        If a model\'s MAE exceeds 2× the average MAE across all models,\n        its weight is halved and the excess redistributed.\n\n        Args:\n            model_name: Name of the model.\n            mae: Recent mean absolute error.\n        """\n        if model_name not in self._recent_errors:\n            self._recent_errors[model_name] = []\n\n        self._recent_errors[model_name].append(mae)\n        # Keep only last 50 entries\n        self._recent_errors[model_name] = self._recent_errors[model_name][-50:]\n\n        # Dynamic adjustment\n        all_maes = []\n        for errors in self._recent_errors.values():\n            if errors:\n                all_maes.append(np.mean(errors))\n\n        if len(all_maes) < 2:\n            return\n\n        avg_mae = np.mean(all_maes)\n        model_avg = np.mean(self._recent_errors[model_name])\n\n        if model_avg > 2 * avg_mae:\n            old_weight = self.weights.get(model_name, 0.0)\n            reduction = old_weight / 2.0\n            self.weights[model_name] = old_weight - reduction\n\n            # Redistribute\n            other_models = [n for n in self.models if n != model_name]\n            if other_models:\n                bonus = reduction / len(other_models)\n                for other in other_models:\n                    self.weights[other] = self.weights.get(other, 0.0) + bonus\n\n            logger.warning(\n                "Model \'{}\' MAE ({:.2f}) > 2× avg ({:.2f}) — weight reduced from {:.3f} to {:.3f}",\n                model_name, model_avg, avg_mae, old_weight, self.weights[model_name],\n            )\n\n    def save(self, directory: str) -> None:\n        """Save all models to a directory.\n\n        Args:\n            directory: Target directory.\n        """\n        os.makedirs(directory, exist_ok=True)\n\n        for model_name, model in self.models.items():\n            model_path = os.path.join(directory, f"{model_name}_model.pkl")\n            try:\n                model.save(model_path)\n            except Exception as exc:\n                logger.error("Failed to save model \'{}\': {}", model_name, exc)\n\n        # Save weights\n        import json\n        weights_path = os.path.join(directory, "ensemble_weights.json")\n        with open(weights_path, "w") as f:\n            json.dump(self.weights, f, indent=2)\n\n        logger.info("Ensemble saved to {}", directory)\n\n    @classmethod\n    def load(cls, directory: str) -> "EnsembleModel":\n        """Load an ensemble from a directory.\n\n        Args:\n            directory: Directory containing saved models.\n\n        Returns:\n            Loaded EnsembleModel instance.\n        """\n        import json\n        from models.arima_model import ARIMAModel\n        from models.xgboost_model import XGBoostModel\n        from models.lstm_model import LSTMModel\n        from models.queuing_model import QueuingModel\n\n        ensemble = cls()\n\n        # Load weights\n        weights_path = os.path.join(directory, "ensemble_weights.json")\n        if os.path.exists(weights_path):\n            with open(weights_path) as f:\n                ensemble.weights = json.load(f)\n\n        # Load each model\n        model_classes: Dict[str, type] = {\n            "arima": ARIMAModel,\n            "xgboost": XGBoostModel,\n            "lstm": LSTMModel,\n            "queuing": QueuingModel,\n        }\n\n        for model_name, model_cls in model_classes.items():\n            model_path = os.path.join(directory, f"{model_name}_model.pkl")\n            if os.path.exists(model_path):\n                try:\n                    model = model_cls.load(model_path)\n                    ensemble.add_model(model)\n                except Exception as exc:\n                    logger.error("Failed to load model \'{}\': {}", model_name, exc)\n            else:\n                logger.warning("Model file not found: {}", model_path)\n\n        return ensemble\n',    "collector/mock_generator.py": '"""Synthetic network telemetry generator for development and testing.\n\nProduces realistic mock data including daily/weekly traffic patterns,\ncongestion events with exponential decay, and correlated packet loss\nand jitter signals.\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\nfrom loguru import logger\n\n\ndef generate_mock_telemetry(\n    n_days: int = 30,\n    interval_sec: float = 1.0,\n    seed: int = 42,\n) -> pd.DataFrame:\n    """Generate a synthetic network telemetry DataFrame.\n\n    The generated data simulates realistic network behaviour:\n    - Base RTT ~20 ms with Gaussian noise (std=3 ms).\n    - Daily pattern: +10 ms during 09:00-18:00, +5 ms during 18:00-22:00.\n    - Weekly pattern: weekends have 30 % lower traffic.\n    - Congestion events: ~5 per day, lasting 2–8 minutes, RTT spikes\n      +50–200 ms with exponential decay back to baseline.\n    - Packet loss: 0.1 % normally, 5–15 % during congestion.\n    - Jitter correlated with congestion.\n\n    Args:\n        n_days: Number of days of data to generate.\n        interval_sec: Sampling interval in seconds.\n        seed: Random seed for reproducibility.\n\n    Returns:\n        DataFrame with columns: timestamp, rtt_ms, packet_loss, jitter_ms,\n        bytes_per_sec, packets_per_sec, congestion_label.\n    """\n    rng = np.random.default_rng(seed)\n\n    total_seconds = int(n_days * 86400)\n    n_samples = int(total_seconds / interval_sec)\n\n    logger.info(\n        "Generating {} samples ({} days, {}s interval)",\n        n_samples, n_days, interval_sec,\n    )\n\n    # ── Timestamps (UTC epoch) ─────────────────────────────────\n    start_ts = 1_700_000_000.0  # arbitrary fixed start\n    timestamps = np.arange(n_samples) * interval_sec + start_ts\n\n    # ── Base RTT with Gaussian noise ───────────────────────────\n    base_rtt = 20.0\n    noise_std = 3.0\n    rtt = base_rtt + rng.normal(0, noise_std, n_samples)\n\n    # ── Daily pattern ──────────────────────────────────────────\n    seconds_in_day = (timestamps - start_ts) % 86400\n    hours = seconds_in_day / 3600.0\n\n    daily_boost = np.where(\n        (hours >= 9) & (hours < 18),\n        10.0,\n        np.where((hours >= 18) & (hours < 22), 5.0, 0.0),\n    )\n    rtt += daily_boost\n\n    # ── Weekly pattern (weekends = 30 % lower traffic) ────────\n    days_from_start = ((timestamps - start_ts) / 86400).astype(int)\n    day_of_week = days_from_start % 7  # 0=Mon … 6=Sun\n    is_weekend = (day_of_week >= 5).astype(float)\n    rtt -= is_weekend * 0.3 * base_rtt  # lower RTT on weekends\n\n    # ── Base traffic metrics ───────────────────────────────────\n    base_bps = 1_000_000.0  # 1 MB/s baseline\n    base_pps = 1000.0  # 1000 packets/s baseline\n    traffic_mult = 1.0 + daily_boost / 20.0 - is_weekend * 0.3\n\n    bytes_per_sec = (\n        base_bps * traffic_mult + rng.normal(0, 50_000, n_samples)\n    )\n    bytes_per_sec = np.clip(bytes_per_sec, 10_000, None)\n\n    packets_per_sec = (\n        base_pps * traffic_mult + rng.normal(0, 50, n_samples)\n    )\n    packets_per_sec = np.clip(packets_per_sec, 10, None)\n\n    # ── Congestion events ──────────────────────────────────────\n    congestion_label = np.zeros(n_samples, dtype=int)\n    n_events = int(n_days * 5)\n\n    for _ in range(n_events):\n        # Random start point\n        event_start = rng.integers(0, max(1, n_samples - 500))\n        duration_sec = rng.integers(120, 480)  # 2–8 minutes\n        duration_samples = int(duration_sec / interval_sec)\n        spike_amplitude = rng.uniform(50, 200)\n        decay_rate = rng.uniform(0.005, 0.02)\n\n        event_end = min(event_start + duration_samples, n_samples)\n        event_range = np.arange(event_end - event_start)\n\n        # Exponential decay from spike amplitude\n        spike = spike_amplitude * np.exp(-decay_rate * event_range)\n        rtt[event_start:event_end] += spike\n        congestion_label[event_start:event_end] = 1\n\n        # Traffic also spikes during congestion\n        bytes_per_sec[event_start:event_end] *= 1.5\n        packets_per_sec[event_start:event_end] *= 1.3\n\n    # ── Packet loss ────────────────────────────────────────────\n    normal_loss_prob = 0.001\n    congestion_loss_prob = rng.uniform(0.05, 0.15, n_samples)\n    loss_prob = np.where(\n        congestion_label == 1, congestion_loss_prob, normal_loss_prob\n    )\n    packet_loss = rng.random(n_samples) < loss_prob\n\n    # ── Jitter (correlated with congestion) ────────────────────\n    base_jitter = 1.0\n    congestion_jitter = rng.uniform(5, 30, n_samples)\n    jitter_ms = np.where(\n        congestion_label == 1,\n        congestion_jitter,\n        base_jitter + rng.exponential(0.5, n_samples),\n    )\n\n    # ── Ensure RTT > 0 ────────────────────────────────────────\n    rtt = np.clip(rtt, 0.1, None)\n\n    df = pd.DataFrame(\n        {\n            "timestamp": timestamps,\n            "rtt_ms": np.round(rtt, 3),\n            "packet_loss": packet_loss.astype(int),\n            "jitter_ms": np.round(jitter_ms, 3),\n            "bytes_per_sec": np.round(bytes_per_sec, 1),\n            "packets_per_sec": np.round(packets_per_sec, 1),\n            "congestion_label": congestion_label,\n        }\n    )\n\n    logger.info(\n        "Generated {} rows | congestion={:.1%} | mean_rtt={:.1f}ms",\n        len(df),\n        congestion_label.mean(),\n        rtt.mean(),\n    )\n\n    return df\n',    "training/dataset.py": '"""Dataset loader with target creation and time-based splits.\n\nProvides ``LatencyDataset`` for loading telemetry data, creating\nshifted target columns, and producing train / val / test splits that\navoid temporal data leakage.\n"""\n\nfrom __future__ import annotations\n\nfrom typing import Tuple\n\nimport numpy as np\nimport pandas as pd\nfrom loguru import logger\n\n\nclass LatencyDataset:\n    """Manages telemetry data for model training.\n\n    Handles CSV / DataFrame loading, target column creation (shifted\n    RTT and congestion labels), and chronological train/val/test\n    splitting.\n    """\n\n    def __init__(self) -> None:\n        self.df: pd.DataFrame = pd.DataFrame()\n\n    def load_from_csv(self, path: str) -> pd.DataFrame:\n        """Load telemetry from a CSV file.\n\n        Args:\n            path: Path to the CSV file.\n\n        Returns:\n            Loaded DataFrame.\n        """\n        self.df = pd.read_csv(path)\n        logger.info("Loaded {} rows from {}", len(self.df), path)\n        return self.df\n\n    def load_from_dataframe(self, df: pd.DataFrame) -> pd.DataFrame:\n        """Load telemetry from an existing DataFrame.\n\n        Args:\n            df: Source DataFrame.\n\n        Returns:\n            The same DataFrame (stored internally).\n        """\n        self.df = df.copy()\n        logger.info("Loaded {} rows from DataFrame", len(self.df))\n        return self.df\n\n    @staticmethod\n    def create_targets(df: pd.DataFrame) -> pd.DataFrame:\n        """Add shifted target columns for supervised learning.\n\n        Creates:\n          - rtt_10s_ahead: rtt_ms shifted backward by 10 rows\n          - rtt_30s_ahead: rtt_ms shifted backward by 30 rows\n          - rtt_60s_ahead: rtt_ms shifted backward by 60 rows\n          - congestion_ahead: congestion_label shifted backward by 10 rows\n\n        Rows with NaN targets (at the tail) are dropped.\n\n        Args:\n            df: DataFrame that must contain ``rtt_ms`` and\n                ``congestion_label`` columns.\n\n        Returns:\n            DataFrame with added target columns, tail NaNs dropped.\n        """\n        df = df.copy()\n        df["rtt_10s_ahead"] = df["rtt_ms"].shift(-10)\n        df["rtt_30s_ahead"] = df["rtt_ms"].shift(-30)\n        df["rtt_60s_ahead"] = df["rtt_ms"].shift(-60)\n\n        if "congestion_label" in df.columns:\n            df["congestion_ahead"] = df["congestion_label"].shift(-10)\n        else:\n            df["congestion_ahead"] = 0\n\n        df = df.dropna(subset=[\n            "rtt_10s_ahead", "rtt_30s_ahead", "rtt_60s_ahead", "congestion_ahead"\n        ]).reset_index(drop=True)\n\n        logger.info("Created targets — {} rows remain after dropping NaN tails", len(df))\n        return df\n\n    @staticmethod\n    def train_val_test_split(\n        df: pd.DataFrame,\n        val_ratio: float = 0.15,\n        test_ratio: float = 0.15,\n    ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n        """Split data chronologically (no random shuffle) to prevent leakage.\n\n        Train = first 70 %, Val = next 15 %, Test = last 15 %.\n\n        Args:\n            df: Full DataFrame (should be sorted by timestamp).\n            val_ratio: Fraction for validation.\n            test_ratio: Fraction for testing.\n\n        Returns:\n            Tuple of (train_df, val_df, test_df).\n        """\n        n = len(df)\n        train_end = int(n * (1 - val_ratio - test_ratio))\n        val_end = int(n * (1 - test_ratio))\n\n        train_df = df.iloc[:train_end].reset_index(drop=True)\n        val_df = df.iloc[train_end:val_end].reset_index(drop=True)\n        test_df = df.iloc[val_end:].reset_index(drop=True)\n\n        logger.info(\n            "Split: train={}, val={}, test={}",\n            len(train_df), len(val_df), len(test_df),\n        )\n        return train_df, val_df, test_df\n',    "training/evaluator.py": '"""Evaluation metrics for latency regression and congestion classification.\n\nProvides MAE, RMSE, MAPE, R² for each prediction horizon and\nprecision / recall / F1 / AUC-ROC for the congestion classifier.\n"""\n\nfrom __future__ import annotations\n\nfrom typing import Any, Dict\n\nimport numpy as np\nfrom loguru import logger\nfrom sklearn.metrics import (\n    average_precision_score,\n    f1_score,\n    mean_absolute_error,\n    mean_squared_error,\n    precision_score,\n    r2_score,\n    recall_score,\n    roc_auc_score,\n)\n\n\nclass Evaluator:\n    """Computes regression and classification metrics."""\n\n    @staticmethod\n    def evaluate_regression(\n        y_true: np.ndarray, y_pred: np.ndarray\n    ) -> Dict[str, Dict[str, float]]:\n        """Evaluate regression metrics per horizon.\n\n        Args:\n            y_true: Ground truth array of shape (n, 3).\n            y_pred: Predictions of shape (n, 3).\n\n        Returns:\n            Dict mapping horizon label to metric dict (mae, rmse, mape, r2).\n        """\n        horizons = ["10s", "30s", "60s"]\n        results: Dict[str, Dict[str, float]] = {}\n\n        for i, hz in enumerate(horizons):\n            yt = y_true[:, i] if y_true.ndim == 2 else y_true\n            yp = y_pred[:, i] if y_pred.ndim == 2 else y_pred\n\n            mae = float(mean_absolute_error(yt, yp))\n            rmse = float(np.sqrt(mean_squared_error(yt, yp)))\n\n            # MAPE — guard against division by zero\n            mask = np.abs(yt) > 1e-6\n            mape = float(np.mean(np.abs((yt[mask] - yp[mask]) / yt[mask])) * 100) if mask.any() else 0.0\n            r2 = float(r2_score(yt, yp))\n\n            results[hz] = {\n                "mae": round(mae, 4),\n                "rmse": round(rmse, 4),\n                "mape": round(mape, 4),\n                "r2": round(r2, 4),\n            }\n\n        return results\n\n    @staticmethod\n    def evaluate_congestion(\n        y_true: np.ndarray, y_pred_proba: np.ndarray, threshold: float = 0.5\n    ) -> Dict[str, float]:\n        """Evaluate binary congestion classification.\n\n        Args:\n            y_true: Ground truth labels (0 or 1).\n            y_pred_proba: Predicted probabilities.\n            threshold: Classification threshold.\n\n        Returns:\n            Dict with precision, recall, f1, auc_roc, average_precision.\n        """\n        y_pred_label = (y_pred_proba >= threshold).astype(int)\n        y_true_int = y_true.astype(int)\n\n        metrics: Dict[str, float] = {}\n\n        metrics["precision"] = round(float(precision_score(y_true_int, y_pred_label, zero_division=0)), 4)\n        metrics["recall"] = round(float(recall_score(y_true_int, y_pred_label, zero_division=0)), 4)\n        metrics["f1"] = round(float(f1_score(y_true_int, y_pred_label, zero_division=0)), 4)\n\n        try:\n            metrics["auc_roc"] = round(float(roc_auc_score(y_true_int, y_pred_proba)), 4)\n        except ValueError:\n            metrics["auc_roc"] = 0.0\n\n        try:\n            metrics["average_precision"] = round(float(average_precision_score(y_true_int, y_pred_proba)), 4)\n        except ValueError:\n            metrics["average_precision"] = 0.0\n\n        return metrics\n\n    @staticmethod\n    def evaluate_all(\n        y_true_rtt: np.ndarray,\n        y_pred_rtt: np.ndarray,\n        y_true_cong: np.ndarray,\n        y_pred_cong_proba: np.ndarray,\n    ) -> Dict[str, Any]:\n        """Combined evaluation report for regression + classification.\n\n        Args:\n            y_true_rtt: True RTT values (n, 3).\n            y_pred_rtt: Predicted RTT values (n, 3).\n            y_true_cong: True congestion labels.\n            y_pred_cong_proba: Predicted congestion probabilities.\n\n        Returns:\n            Combined report dict with \'regression\' and \'congestion\' keys.\n        """\n        regression = Evaluator.evaluate_regression(y_true_rtt, y_pred_rtt)\n        congestion = Evaluator.evaluate_congestion(y_true_cong, y_pred_cong_proba)\n\n        report = {\n            "regression": regression,\n            "congestion": congestion,\n        }\n\n        logger.info("Evaluation report: {}", report)\n        return report\n',    "training/trainer.py": '"""Training orchestrator — runs the full train → evaluate → log pipeline.\n\nTrains all four models (ARIMA, XGBoost, LSTM, Queuing), evaluates on\nthe test set, logs everything to MLflow, and saves the best ensemble.\n"""\n\nfrom __future__ import annotations\n\nimport os\nimport time\nfrom typing import Any, Dict, List, Optional\n\nimport numpy as np\nimport pandas as pd\nfrom loguru import logger\n\nfrom config.settings import get_settings\nfrom features.pipeline import FeaturePipeline\nfrom models.arima_model import ARIMAModel\nfrom models.xgboost_model import XGBoostModel\nfrom models.lstm_model import LSTMModel\nfrom models.queuing_model import QueuingModel\nfrom models.ensemble import EnsembleModel\nfrom training.dataset import LatencyDataset\nfrom training.evaluator import Evaluator\n\ntry:\n    import mlflow\n    MLFLOW_AVAILABLE = True\nexcept ImportError:\n    MLFLOW_AVAILABLE = False\n\n\nclass Trainer:\n    """Orchestrates the end-to-end training pipeline.\n\n    Steps:\n      1. Feature pipeline transform\n      2. Create train / val / test splits\n      3. Train all 4 models sequentially\n      4. Evaluate each on the test set\n      5. Log to MLflow (params, metrics, artifacts)\n      6. Save ensemble to ``MODEL_DIR``\n      7. Return evaluation report\n    """\n\n    def __init__(self) -> None:\n        self.settings = get_settings()\n        self.pipeline = FeaturePipeline()\n        self.evaluator = Evaluator()\n\n    def _get_feature_columns(self, df: pd.DataFrame) -> List[str]:\n        """Determine feature columns (exclude targets and metadata).\n\n        Args:\n            df: Enriched DataFrame with features + targets.\n\n        Returns:\n            List of feature column names.\n        """\n        exclude = {\n            "timestamp", "rtt_ms", "packet_loss", "jitter_ms",\n            "bytes_per_sec", "packets_per_sec", "congestion_label",\n            "rtt_10s_ahead", "rtt_30s_ahead", "rtt_60s_ahead",\n            "congestion_ahead",\n        }\n        return [c for c in df.columns if c not in exclude]\n\n    def run(self, df: pd.DataFrame) -> Dict[str, Any]:\n        """Execute the full training pipeline.\n\n        Args:\n            df: Raw telemetry DataFrame.\n\n        Returns:\n            Evaluation report dictionary.\n        """\n        start_time = time.time()\n        logger.info("=" * 60)\n        logger.info("TRAINING PIPELINE START")\n        logger.info("=" * 60)\n\n        # 1. Feature engineering\n        logger.info("Step 1/6: Feature engineering")\n        enriched = self.pipeline.transform(df)\n\n        # 2. Create targets and split\n        logger.info("Step 2/6: Creating targets and splits")\n        dataset = LatencyDataset()\n        enriched = dataset.create_targets(enriched)\n        train_df, val_df, test_df = dataset.train_val_test_split(enriched)\n\n        feature_cols = self._get_feature_columns(enriched)\n        target_cols_rtt = ["rtt_10s_ahead", "rtt_30s_ahead", "rtt_60s_ahead"]\n        target_col_jitter = "jitter_ms" if "jitter_ms" in train_df.columns else None\n        target_col_cong = "congestion_ahead"\n\n        X_train = train_df[feature_cols].values.astype(np.float32)\n        X_val = val_df[feature_cols].values.astype(np.float32)\n        X_test = test_df[feature_cols].values.astype(np.float32)\n\n        y_train_rtt = train_df[target_cols_rtt].values.astype(np.float32)\n        y_val_rtt = val_df[target_cols_rtt].values.astype(np.float32)\n        y_test_rtt = test_df[target_cols_rtt].values.astype(np.float32)\n\n        y_train_cong = train_df[target_col_cong].values.astype(np.float32)\n        y_val_cong = val_df[target_col_cong].values.astype(np.float32)\n        y_test_cong = test_df[target_col_cong].values.astype(np.float32)\n\n        # Replace NaN/inf in features\n        X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)\n        X_val = np.nan_to_num(X_val, nan=0.0, posinf=0.0, neginf=0.0)\n        X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)\n\n        # Construct combined targets for XGBoost (rtt×3 + jitter + congestion)\n        if target_col_jitter and target_col_jitter in train_df.columns:\n            jitter_train = train_df[target_col_jitter].values.astype(np.float32)\n            jitter_val = val_df[target_col_jitter].values.astype(np.float32)\n        else:\n            jitter_train = np.zeros(len(train_df), dtype=np.float32)\n            jitter_val = np.zeros(len(val_df), dtype=np.float32)\n\n        y_xgb_train = np.column_stack([y_train_rtt, jitter_train, y_train_cong])\n        y_xgb_val = np.column_stack([y_val_rtt, jitter_val, y_val_cong])\n\n        # LSTM targets (rtt×3 + congestion)\n        y_lstm_train = np.column_stack([y_train_rtt, y_train_cong])\n        y_lstm_val = np.column_stack([y_val_rtt, y_val_cong])\n\n        # MLflow setup\n        if MLFLOW_AVAILABLE:\n            try:\n                mlflow.set_tracking_uri(self.settings.MLFLOW_TRACKING_URI)\n                mlflow.set_experiment(self.settings.MLFLOW_EXPERIMENT_NAME)\n            except Exception as exc:\n                logger.warning("MLflow setup failed (will continue without): {}", exc)\n\n        reports: Dict[str, Any] = {}\n        ensemble = EnsembleModel()\n\n        mlflow_run = None\n        if MLFLOW_AVAILABLE:\n            try:\n                mlflow_run = mlflow.start_run()\n            except Exception:\n                mlflow_run = None\n\n        try:\n            # 3. Train models\n            logger.info("Step 3/6: Training models")\n\n            # ── ARIMA ──\n            logger.info("Training ARIMA...")\n            arima = ARIMAModel()\n            try:\n                arima_series = train_df["rtt_ms"].values[-2000:]  # Use last 2000 for speed\n                arima.fit(None, arima_series)\n                ensemble.add_model(arima)\n                logger.info("ARIMA trained successfully")\n            except Exception as exc:\n                logger.error("ARIMA training failed: {}", exc)\n\n            # ── XGBoost ──\n            logger.info("Training XGBoost...")\n            xgb_model = XGBoostModel()\n            try:\n                xgb_model.fit(\n                    X_train, y_xgb_train,\n                    eval_set=[(X_val, y_xgb_val)],\n                    feature_names=feature_cols,\n                )\n                ensemble.add_model(xgb_model)\n                logger.info("XGBoost trained successfully")\n            except Exception as exc:\n                logger.error("XGBoost training failed: {}", exc)\n\n            # ── LSTM ──\n            logger.info("Training LSTM...")\n            lstm_model = LSTMModel(n_features=X_train.shape[1])\n            try:\n                lstm_model.fit(\n                    X_train, y_lstm_train,\n                    X_val=X_val, y_val=y_lstm_val,\n                )\n                ensemble.add_model(lstm_model)\n                logger.info("LSTM trained successfully")\n            except Exception as exc:\n                logger.error("LSTM training failed: {}", exc)\n\n            # ── Queuing ──\n            logger.info("Training Queuing Model...")\n            queuing = QueuingModel()\n            try:\n                # Use packets_per_sec and jitter as features\n                pps_idx = feature_cols.index("packets_per_sec") if "packets_per_sec" in feature_cols else None\n                jit_idx = feature_cols.index("jitter_ms") if "jitter_ms" in feature_cols else None\n\n                if pps_idx is not None and jit_idx is not None:\n                    q_train = X_train[:, [pps_idx, jit_idx]]\n                elif pps_idx is not None:\n                    q_train = X_train[:, [pps_idx]]\n                else:\n                    q_train = X_train[:, :2]\n\n                queuing.fit(q_train, train_df["rtt_ms"].values)\n                ensemble.add_model(queuing)\n                logger.info("Queuing model trained successfully")\n            except Exception as exc:\n                logger.error("Queuing model training failed: {}", exc)\n\n            # 4. Evaluate\n            logger.info("Step 4/6: Evaluating models")\n            for model_name, model in ensemble.models.items():\n                try:\n                    if model_name == "arima":\n                        pred = model.predict(X_test)\n                        if pred.shape[0] == 1:\n                            pred = np.broadcast_to(pred, (len(X_test), 3)).copy()\n                        cong_pred = model.predict_proba_congestion(X_test)\n                        if cong_pred.shape[0] == 1:\n                            cong_pred = np.broadcast_to(cong_pred, (len(X_test),)).copy()\n                    elif model_name == "queuing":\n                        if pps_idx is not None and jit_idx is not None:\n                            q_test = X_test[:, [pps_idx, jit_idx]]\n                        elif pps_idx is not None:\n                            q_test = X_test[:, [pps_idx]]\n                        else:\n                            q_test = X_test[:, :2]\n                        pred = model.predict(q_test)\n                        cong_pred = model.predict_proba_congestion(q_test)\n                    else:\n                        pred = model.predict(X_test)\n                        cong_pred = model.predict_proba_congestion(X_test)\n\n                    # Ensure shapes match\n                    pred = pred[:len(y_test_rtt)]\n                    cong_pred = cong_pred[:len(y_test_cong)]\n\n                    report = self.evaluator.evaluate_all(\n                        y_test_rtt[:len(pred)], pred,\n                        y_test_cong[:len(cong_pred)], cong_pred,\n                    )\n                    reports[model_name] = report\n\n                    # Log to MLflow\n                    if MLFLOW_AVAILABLE and mlflow_run:\n                        try:\n                            for hz, metrics in report["regression"].items():\n                                for metric, value in metrics.items():\n                                    mlflow.log_metric(f"{model_name}_{hz}_{metric}", value)\n                            for metric, value in report["congestion"].items():\n                                mlflow.log_metric(f"{model_name}_congestion_{metric}", value)\n                        except Exception:\n                            pass\n\n                except Exception as exc:\n                    logger.error("Evaluation of \'{}\' failed: {}", model_name, exc)\n\n            # 5. Ensemble evaluation\n            logger.info("Step 5/6: Ensemble evaluation")\n            ensemble_pred = ensemble.predict(X_test)\n\n            # 6. Save ensemble\n            logger.info("Step 6/6: Saving models")\n            model_dir = self.settings.MODEL_DIR\n            ensemble.save(model_dir)\n\n            if MLFLOW_AVAILABLE and mlflow_run:\n                try:\n                    mlflow.log_param("n_train", len(train_df))\n                    mlflow.log_param("n_val", len(val_df))\n                    mlflow.log_param("n_test", len(test_df))\n                    mlflow.log_param("n_features", len(feature_cols))\n                    mlflow.log_param("models_trained", list(ensemble.models.keys()))\n                except Exception:\n                    pass\n\n        finally:\n            if MLFLOW_AVAILABLE and mlflow_run:\n                try:\n                    mlflow.end_run()\n                except Exception:\n                    pass\n\n        elapsed = time.time() - start_time\n        logger.info("TRAINING COMPLETE in {:.1f}s", elapsed)\n        logger.info("Model reports: {}", reports)\n\n        return {\n            "models": reports,\n            "ensemble_prediction_sample": ensemble_pred,\n            "training_time_s": round(elapsed, 2),\n            "n_features": len(feature_cols),\n            "feature_names": feature_cols,\n        }\n',}for path, content in modules.items():    full = os.path.join(BASE, path)    os.makedirs(os.path.dirname(full), exist_ok=True)    with open(full, 'w') as f:        f.write(content)    print(f"  ✅ {path}")print(f"\n✅ All {len(modules)} source modules written")

In [ ]:
# @title 5️⃣ Generate Mock Training Dataimport os, gc; os.chdir('/content/netPredictor')import sys; sys.path.insert(0, '/content/netPredictor')# ⚠️ Colab free tier has ~12 GB RAM. 7 days ≈ 605K rows is safe.# Increase to 14-30 only on Colab Pro (25+ GB RAM).DAYS = 7  # @param {type:"integer"}SEED = 42  # @param {type:"integer"}from collector.mock_generator import generate_mock_telemetrydf = generate_mock_telemetry(n_days=DAYS, interval_sec=1.0, seed=SEED)df.to_csv('data/mock_telemetry.csv', index=False)print(f"\n📊 Generated {len(df):,} rows ({DAYS} days)")print(f"   RTT: mean={df['rtt_ms'].mean():.1f}ms, std={df['rtt_ms'].std():.1f}ms")print(f"   Congestion rate: {df['congestion_label'].mean():.1%}")print(f"   File: data/mock_telemetry.csv ({os.path.getsize('data/mock_telemetry.csv')/1e6:.1f} MB)")del df; gc.collect()print("🧹 Memory freed")

In [ ]:
# @title 6️⃣ Train All Models (ARIMA + XGBoost + LSTM + Queuing)import os, gc, timeos.chdir('/content/netPredictor')os.environ['MODEL_DIR'] = 'models/saved'import pandas as pdfrom config.logging_config import setup_loggingfrom training.trainer import Trainersetup_logging(level="INFO", json_output=False)df = pd.read_csv('data/mock_telemetry.csv')# ⚠️ Colab RAM limit! 80K rows is safe for free tier (~12 GB).# The LSTM creates sliding-window arrays of shape (N, 60, ~45) which# use ~N×60×45×4 bytes. At 80K rows this is ~860 MB (manageable).# Increase to 150K-200K only on Colab Pro.MAX_ROWS = 80_000  # @param {type:"integer"}if len(df) > MAX_ROWS:    print(f"Subsampling {len(df):,} → {MAX_ROWS:,} rows")    df = df.tail(MAX_ROWS).reset_index(drop=True)gc.collect()print(f"\n🏋️ Training on {len(df):,} rows...")print(f"   GPU: {('Yes — ' + __import__('torch').cuda.get_device_name(0)) if __import__('torch').cuda.is_available() else 'No (CPU mode)'}")print()trainer = Trainer()start = time.time()report = trainer.run(df)elapsed = time.time() - startdel df; gc.collect()print(f"\n✅ Training complete in {elapsed:.1f}s")

In [ ]:
# @title 7️⃣ Training Results & Evaluationprint("=" * 65)print("  MODEL EVALUATION RESULTS")print("=" * 65)max_mae = 0.0for model_name, metrics in report["models"].items():    print(f"\n  📊 {model_name.upper()}")    if "regression" in metrics:        print(f"  {'Horizon':<10} {'MAE (ms)':<12} {'RMSE (ms)':<12} {'R²':<10}")        print(f"  {'─'*10} {'─'*12} {'─'*12} {'─'*10}")        for hz, vals in metrics["regression"].items():            mae = vals['mae']            max_mae = max(max_mae, mae)            print(f"  {hz:<10} {mae:<12.2f} {vals['rmse']:<12.2f} {vals['r2']:<10.3f}")    if "congestion" in metrics:        c = metrics["congestion"]        print(f"  Congestion: F1={c['f1']:.3f}  Precision={c['precision']:.3f}  Recall={c['recall']:.3f}")print(f"\n{'=' * 65}")if max_mae > 20.0:    print(f"  ❌ SANITY CHECK FAILED: max MAE = {max_mae:.2f}ms > 20ms")else:    print(f"  ✅ All models passed (max MAE = {max_mae:.2f}ms)")print(f"  ⏱️  Total training time: {report['training_time_s']:.1f}s")print(f"  📐 Features used: {report['n_features']}")print("=" * 65)

In [ ]:
# @title 8️⃣ Save & Download Trained Modelsimport shutil, osfrom datetime import datetimeos.chdir('/content/netPredictor')timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')zip_name = f'trained_models_{timestamp}'# Create zipshutil.make_archive(zip_name, 'zip', 'models/saved')zip_path = f'{zip_name}.zip'print(f"📦 Models packaged: {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)")# Save to Drive if mountedtry:    if DRIVE_DIR:        drive_zip = os.path.join(DRIVE_DIR, os.path.basename(zip_path))        shutil.copy2(zip_path, drive_zip)        # Also copy individual model files        drive_models = os.path.join(DRIVE_DIR, 'latest')        if os.path.exists(drive_models):            shutil.rmtree(drive_models)        shutil.copytree('models/saved', drive_models)        print(f"💾 Saved to Google Drive: {DRIVE_DIR}")        print(f"   ZIP: {drive_zip}")        print(f"   Models: {drive_models}/")except:    pass# Download to local machinefrom google.colab import filesfiles.download(zip_path)print("\n⬇️  Downloading to your local machine...")print("\n📋 To use locally, run:")print(f"   python scripts/sync_models.py --import {zip_name}.zip")